# Experimento do Artigo Técnico — Árvore de Decisão vs. Random Forest vs. XGBoost

**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)  
**Equipe (Grupo 42):** Wendell Rodrigues Macedo, Vinicius Firmino Souto, Fernando Paiva Coelho, João Gustavo Sanches, Daniel Dias Paranhos  
**Disciplina:** Estrutura de Dados II — Profa. Dra. Andréa Ono Sakai

Este notebook reproduz integralmente o experimento comparativo do Artigo Técnico:
- **Mesma base e divisão temporal sem vazamento:** Treino (`80.386` linhas), Validação (`32.148` linhas para ajuste de hiperparâmetros) e Teste (`48.270` linhas);
- **Três modelos ajustados de forma justa:** Árvore de Decisão (CART), Random Forest (Bagging) e XGBoost (Gradient Boosted Trees);
- **Métricas avaliadas:** Precisão, Recall e F1 por classe (`OK`, `RISCO`, `FALHA`), `F1 Macro`, Matrizes de Confusão 3×3, Importância dos Atributos e Tempos de Treino/Predição.

In [1]:
import json
from pathlib import Path
from src.experimento_artigo import executar_experimento_artigo

dir_proj = Path('..').resolve()
with open(dir_proj / 'metadados' / 'resumo_artigo_comparativo.json', encoding='utf-8') as f:
    ra = json.load(f)

print('=== COMPARAÇÃO NO TESTE (teste_periodo_b.csv — 48.270 medições, random_state=42) ===')
for k in ['arvore_decisao', 'random_forest', 'xgboost']:
    m = ra['comparacao_teste_principal'][k]
    mt = m['metricas_teste']
    print(f"- {m['nome']:34s} | F1 Macro={mt['f1_macro']:.4f} | Acurácia={mt['acuracia']:.4f} | Recall FALHA={mt['por_classe']['FALHA']['recall']:.4f} | Recall RISCO={mt['por_classe']['RISCO']['recall']:.4f} | Erros={mt['total_erros']:3d} | Treino={m['tempo_treino_ms']:8.2f} ms | Pred={m['tempo_pred_teste_ms']:6.2f} ms ({m['tempo_pred_us_por_amostra']:.4f} us/amostra)")

=== COMPARAÇÃO NO TESTE (teste_periodo_b.csv — 48.270 medições, random_state=42) ===
- Árvore de Decisão (CART)           | F1 Macro=1.0000 | Acurácia=1.0000 | Recall FALHA=1.0000 | Recall RISCO=1.0000 | Erros=  0 | Treino=  844.61 ms | Pred=  8.52 ms (0.1765 us/amostra)
- Random Forest (Bagging)            | F1 Macro=0.7544 | Acurácia=0.9967 | Recall FALHA=1.0000 | Recall RISCO=0.1534 | Erros=161 | Treino=18779.15 ms | Pred=231.55 ms (4.7970 us/amostra)
- XGBoost (Gradient Boosted Trees)   | F1 Macro=1.0000 | Acurácia=1.0000 | Recall FALHA=1.0000 | Recall RISCO=1.0000 | Erros=  0 | Treino=52649.35 ms | Pred=361.83 ms (7.4960 us/amostra)


In [2]:
# Matrizes de Confusão 3x3 e Importância dos Atributos dos 3 Modelos no Teste
print(json.dumps(ra['comparacao_teste_principal'], indent=2, ensure_ascii=False))

{
  "arvore_decisao": {
    "nome": "Árvore de Decisão (CART)",
    "hiperparametros": "criterion='entropy', max_depth=6, min_samples_leaf=20, random_state=42",
    "estrutura": "1 árvore (7 folhas, 13 nós totais)",
    "tempo_treino_ms": 844.61,
    "tempo_pred_teste_ms": 8.52,
    "tempo_pred_us_por_amostra": 0.1765,
    "metricas_teste": {
      "matriz_confusao": {
        "OK": {
          "OK": 28303,
          "RISCO": 0,
          "FALHA": 0
        },
        "RISCO": {
          "OK": 0,
          "RISCO": 189,
          "FALHA": 0
        },
        "FALHA": {
          "OK": 0,
          "RISCO": 0,
          "FALHA": 19778
        }
      },
      "por_classe": {
        "OK": {
          "precisao": 1.0,
          "recall": 1.0,
          "f1": 1.0,
          "suporte": 28303
        },
        "RISCO": {
          "precisao": 1.0,
          "recall": 1.0,
          "f1": 1.0,
          "suporte": 189
        },
        "FALHA": {
          "precisao": 1.0,
          "rec